In [34]:
# Imports

from datasets import load_from_disk
import numpy as np
from transformers import (
	AutoModelForMaskedLM,
	AutoModelForSequenceClassification,
	AutoTokenizer,
	Trainer,
	TrainingArguments,
    DataCollatorForLanguageModeling
)
from sklearn.metrics import (
	accuracy_score, 
    precision_recall_fscore_support, 
    confusion_matrix, 
    ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

In [35]:
# constants

large_dataset_path = "data/my_saved_dataset_large"
base_model_path = "data/my_saved_model"
unlabaled_trained_model_path = "data/unlabeled_trained_model"
text_column = "text"

In [36]:
# parameters

seed = 42
num_samples_large_dataset = 5000

large_dataset_epoch = 1
large_dataset_per_device_train_batch_size = 8
large_dataset_logging_steps = 50


stratify_test_size = 0.5

small_dataset_eval_steps = 10
small_dataset_num_train_epochs = 1
small_dataset_per_device_train_batch_size = 4

In [37]:
# Load and prepare dataset for unsupervised training

print("Loading large dataset...")
large_dataset = load_from_disk(large_dataset_path)
large_dataset = large_dataset.shuffle(seed=seed)
if num_samples_large_dataset is not None:
    large_dataset = large_dataset.select(range(num_samples_large_dataset))

In [38]:
# Load tokenizer and model for unsupervised training

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(base_model_path)

print("Loading base model...")
model = AutoModelForMaskedLM.from_pretrained(base_model_path).to("xpu")

In [39]:
# Tokenize the text column of the dataset
def tokenize_batch_large_dataset(batch):
	return tokenizer(
		batch[text_column]	
		, truncation=True,
		max_length=128,
	)

print("Tokenizing text column...")
tokenized_dataset = large_dataset.map(tokenize_batch_large_dataset, batched=True, remove_columns=large_dataset.column_names)


In [40]:
# Prepare data collator for masked language modeling

data_collator = DataCollatorForLanguageModeling( tokenizer=tokenizer )

In [41]:
# Set up Trainer for unsupervised training

training_args = TrainingArguments(
	num_train_epochs=large_dataset_epoch,
	logging_steps=large_dataset_logging_steps if large_dataset_logging_steps is not None else 500,
    per_device_train_batch_size=large_dataset_per_device_train_batch_size,
	remove_unused_columns=True
)

trainer = Trainer(
	model=model,
	args=training_args,
	train_dataset=tokenized_dataset,
	data_collator=data_collator
)

In [42]:
# Train the model on the unlabeled text data

print("Starting training on unlabeled text...")
trainer.train()
print("Finished unsupervised training.")
unlabaled_trained_model = model.to("xpu")
unlabaled_trained_model.save_pretrained(unlabaled_trained_model_path)


In [ ]:
# Load and prepare small dataset for supervised training and testing

small_dataset_path = "data/my_saved_dataset_small"
print("Loading small dataset...")
small_dataset = load_from_disk(small_dataset_path)
combo_column = "combo"
small_dataset = small_dataset.map(
	lambda example: {
		combo_column: f"{example['politics'][0]}{example['sentiment'][0]}".upper()
	},
    remove_columns=["politics", "sentiment"]
)
small_dataset = small_dataset.class_encode_column(combo_column)
print(small_dataset.features)

In [ ]:
# Split the small dataset into training and testing sets, stratifying by the combo column to maintain class distribution

split_dataset = small_dataset.train_test_split(
	test_size=stratify_test_size,
	seed=seed,
	stratify_by_column=combo_column
)
train_dataset = split_dataset["train"]
test_dataset = split_dataset["test"]
print(f"Train size: {len(train_dataset)} | Test size: {len(test_dataset)}")

In [ ]:
# Tokenize thee text column of the training and testing datasets

def tokenize_with_labels(batch):
    tokens = tokenizer(
        batch["text"],
        truncation=True,
        max_length=128,
        padding="max_length",
    )
    tokens["labels"] = batch[combo_column]
    return tokens

train_tokens = train_dataset.map(
	tokenize_with_labels,
	batched=True,
	remove_columns=train_dataset.column_names
)
test_tokens = test_dataset.map(
    tokenize_with_labels,
    batched=True,
    remove_columns=test_dataset.column_names
)

In [ ]:
# Make the model for supervised training

label2id = {
    "LP": 0,
    "LN": 1,
    "RP": 2,
    "RN": 3
}

clf_model = AutoModelForSequenceClassification.from_pretrained(
    unlabaled_trained_model_path,
    label2id=label2id,
    id2label={value: key for key, value in label2id.items()},
    ignore_mismatched_sizes=True,
)

In [ ]:
# Make the trainer for supervised training, with custom evaluation metrics

cls_args = TrainingArguments(
    eval_strategy="steps",
    eval_steps=small_dataset_eval_steps,
	num_train_epochs=small_dataset_num_train_epochs,
    per_device_train_batch_size=small_dataset_per_device_train_batch_size
)

# source for base code is https://medium.com/@rakeshrajpurohit/customized-evaluation-metrics-with-hugging-face-trainer-3ff00d936f99
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted"
    )

    accuracy = accuracy_score(labels, predictions)

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1,
    }

pred_trainer = Trainer(
	model=clf_model,
	args=cls_args,
	train_dataset=train_tokens,
	eval_dataset=test_tokens,
    compute_metrics=compute_metrics
)

In [ ]:
# Train the model on the labeled data and evaluate

print("Starting training with metrics...")
pred_trainer.train()

In [ ]:
# Calculate and display the confusion matrix for the test set predictions

print("Calculating confusion matrix...")
pred_output = pred_trainer.predict(test_tokens)
predictions = np.argmax(pred_output.predictions, axis=-1)
true_labels = np.array(test_tokens["labels"])
cm = confusion_matrix(true_labels, predictions)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['LP', 'LN', 'RP', 'RN'])
disp.plot()
plt.show()

In [ ]:
# Display metrics

loss = []
accuracy = []
precision = []
recall = []
f1 = []

for record in pred_trainer.state.log_history:
    # eval_loss is not present in the final record, when the measured metrics are no longer there
    if 'eval_loss' not in record:
        break
    loss.append(record["eval_loss"])
    accuracy.append(record["eval_accuracy"])
    precision.append(record["eval_precision"])
    recall.append(record["eval_recall"])
    f1.append(record["eval_f1"])
metrics = {
    "Loss": loss,
    "Accuracy": accuracy,
    "Precision": precision,
    "Recall": recall,
    "F1 Score": f1
}

plt.figure(figsize=(9, 6))
for metric_name, metric_values in metrics.items():
    plt.plot(metric_values, label=metric_name)
plt.xlabel("Evaluation Steps")
plt.ylabel("Metric Value")
plt.title("Evaluation Metrics Over Time")
plt.legend()
plt.grid()
plt.show()

for metric_name, metric_values in metrics.items():
    plt.figure(figsize=(9, 6))
    plt.plot(metric_values)  
    plt.xlabel("Evaluation Steps")
    plt.title(f"{metric_name} Over Time")
    plt.ylabel(metric_name)
    plt.title(f"{metric_name} Over Time")
    plt.grid()
    plt.show()